In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


In [ ]:

DATA_DIR = "/path/to/local/resource"
IMG_SIZE = (224, 224)
BATCH = 32
EPOCHS = 8
SEED = 1337

train_ds = tf.keras.utils.image_dataset_from_directory(
    f"{DATA_DIR}/train",
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH,
    seed=SEED,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    f"{DATA_DIR}/val",
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH,
    seed=SEED,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    f"{DATA_DIR}/test",
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH,
    shuffle=False,
)

print("Classes:", train_ds.class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(2000).prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)
test_ds  = test_ds.cache().prefetch(AUTOTUNE)



In [ ]:
# Augmentation 
data_aug = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
])

In [ ]:

# Backbone Fast + Strong (like X-ray style transfer learning)

base = keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet"
)
base.trainable = False

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = data_aug(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(2, activation="softmax")(x)
model = keras.Model(inputs, outputs)



In [ ]:

model.compile(
    optimizer=keras.optimizers.Adam(3e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cbs = [
    keras.callbacks.ModelCheckpoint("ct_best.keras", save_best_only=True, monitor="val_accuracy", mode="max"),
    keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True, monitor="val_accuracy", mode="max"),
]

model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=cbs)

print("\nEvaluating on test...")
test_loss, test_acc = model.evaluate(test_ds)
print("Test acc:", test_acc)

model.save("ct_final.keras")
print("Saved: ct_final.keras")


In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve
import matplotlib.pyplot as plt

DATA_DIR = "/path/to/local/resource"
IMG_SIZE = (224, 224)
BATCH = 32

model = tf.keras.models.load_model("ct_best.keras")

test_ds = tf.keras.utils.image_dataset_from_directory(
    f"{DATA_DIR}/test",
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH,
    shuffle=False,
)

y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)
probs = model.predict(test_ds, verbose=0)
y_pred = probs.argmax(axis=1)
y_score = probs[:, 1]

print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))
print("\nReport:\n", classification_report(y_true, y_pred, digits=4))

auc = roc_auc_score(y_true, y_score)
print("ROC AUC:", auc)

# Confusion matrix plot
cm = confusion_matrix(y_true, y_pred)
plt.figure()
plt.imshow(cm)
plt.title("CT Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
for (i, j), v in np.ndenumerate(cm):
    plt.text(j, i, str(v), ha="center", va="center")
plt.tight_layout()
plt.savefig("ct_confusion_matrix.png", dpi=200)

# ROC plot
fpr, tpr, _ = roc_curve(y_true, y_score)
plt.figure()
plt.plot(fpr, tpr)
plt.title(f"CT ROC Curve (AUC={auc:.3f})")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.tight_layout()
plt.savefig("ct_roc.png", dpi=200)

print("Saved: ct_confusion_matrix.png, ct_roc.png")
